In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls "/content/drive/MyDrive/Colab Notebooks/"

 10685642.zip
 aplication_ai_mlp.ipynb
 app_ai_scaler.pkl
 app_ensemble_threshold.npy
'application_dataset (1).csv.gz'
 application_dataset.csv.gz
 application_gatekeeper.ipynb
 application_gatekeeper_streaming.pkl
 application_logs_kafka.ipynb
 application_test_logs_malicious_heavy.csv
 Application_valid_layer.ipynb
 app_test_logs.csv
'ca (1).pem'
 ca_b.pem
 ca.pem
 client.truststore.jks
'Copy of NF-ToN-IoT-v3.csv'
 cryptography.ipynb
 csic_final.csv
'cybersecurity_threat_detection_logs (1).csv'
 cybersecurity_threat_detection_logs.csv
 dataset_20_percent.csv
 deep_ai_model.h5
 deep_ai_scaler.pkl
 df_refined.csv
 ensemble_threshold.npy
 feature_app_columns.pkl
 feature_columns.pkl
 files_split.ipynb
 forensic_audit.jsonl
 forensic_explanation.ipynb
 forensic_reports.json
 gatekeeper_network.pkl
 kept_logs_for_training.csv
 lgbm_model.pkl
 log_file_1972.csv
 log_files_iot.zip
 meta_model.pkl
 mlp_application_model.h5
 mlp_feature_columns.pkl
 mlp_label_encoders.pkl
 mlp_model.keras
 ml

In [5]:
import pandas as pd
import joblib
import numpy as np
from tensorflow.keras.models import load_model

FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# ===============================
# LOAD DLQ STREAM
# ===============================

df=pd.read_csv(FILE_PATH+"windows_dlq_stream_test.csv")

print("DLQ events:",len(df))

# ===============================
# FEATURE ENGINEERING
# ===============================

df["date_and_time"]=pd.to_datetime(df["date_and_time"])

df["hour"]=df["date_and_time"].dt.hour

df["is_night"]=((df["hour"]<6)|(df["hour"]>22)).astype(int)

features=[

"event_id",
"event_freq",
"burst_count",
"hour",
"is_night"

]

X=df[features]

# ===============================
# LOAD MODEL
# ===============================

model=load_model(FILE_PATH+"windows_mlp_model.h5")

scaler=joblib.load(FILE_PATH+"windows_mlp_scaler.pkl")

X=scaler.transform(X)

# ===============================
# PREDICT
# ===============================

probs=model.predict(X)

df["mlp_anomaly"]=0
df.loc[probs.flatten()>0.6,"mlp_anomaly"]=1

# ===============================
# EXTRACT MLP ANOMALIES
# ===============================

anomalies = df[df["mlp_anomaly"] == 1]

print("\nMLP anomalies detected:", len(anomalies))
print(anomalies.head())

# ===============================
# SAVE ANOMALY FILE
# ===============================

anomalies.to_csv(
FILE_PATH+"windows_mlp_alerts.csv",
index=False
)

# print("MLP alerts saved:",len(anomalies))
# # ===============================
# # PRINT ANOMALIES
# # ===============================

# anomalies=df[df["mlp_anomaly"]==1]

# print("\nMLP anomalies detected:",len(anomalies))

# print(anomalies[[
# "date_and_time",
# "event_id",
# "risk_score",
# "mlp_anomaly"
# ]].head())

# # ===============================
# # SAVE
# # ===============================

#df.to_csv(FILE_PATH+"windows_mlp_output.csv",index=False)

print("MLP analysis completed")

DLQ events: 500


16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

MLP anomalies detected: 492
        date_and_time                               source  event_id  \
3 2024-10-14 23:39:03  Microsoft-Windows-Security-Auditing      4672   
4 2024-10-15 00:39:42  Microsoft-Windows-Security-Auditing      4798   
5 2024-10-15 02:03:30  Microsoft-Windows-Security-Auditing      4672   
6 2024-10-15 02:05:24  Microsoft-Windows-Security-Auditing      4672   
7 2024-10-17 14:17:14  Microsoft-Windows-Security-Auditing      4798   

             task_category                                            message  \
3            Special Logon  Special privileges assigned to new logon.\r\n\...   
4  User Account Management  A user's local group membership was enumerated...   
5            Special Logon  Special privileges assigned to new logon.\r\n\...   
6            Special Logon  Special privileges assigned to new logon.\r\n\...   
7  User Account Management  A user's local group membership was enumerated...   

   hour  nig